In [ ]:
from __future__ import annotations


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


N_OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET_LOG_COL = "Target_Log"
FULL_VIEW_NAME = "full_feature_unified"
TEN_VIEW_NAME = "ten_feature_unified"
WASTE_TYPES = ["AW", "CDW", "IW", "MSW"]
ROUTE_COLUMNS = [f"WasteRoute_{waste_type}" for waste_type in WASTE_TYPES]
FORMAL_TEN_FEATURE_CONTRACT = [
    "NY.GDP.MKTP.PP.KD",
    "SP.POP.TOTL",
    "NY.GDP.PCAP.PP.KD",
    "EN.POP.DNST",
    "Proxy_NY.GDP.MKTP.PP.KD_log1p",
    "Proxy_SP.POP.TOTL_log1p",
    "Proxy_NY.GDP.PCAP.PP.KD_log1p",
    "Proxy_EN.POP.DNST_log1p",
    "EN.GHG.CO2.MT.CE.AR5",
    "Proxy_EN.GHG.CO2.MT.CE.AR5_log1p",
]
PROXY_SOURCE_MAP = {
    "Proxy_NY.GDP.MKTP.PP.KD_log1p": "NY.GDP.MKTP.PP.KD",
    "Proxy_SP.POP.TOTL_log1p": "SP.POP.TOTL",
    "Proxy_NY.GDP.PCAP.PP.KD_log1p": "NY.GDP.PCAP.PP.KD",
    "Proxy_EN.POP.DNST_log1p": "EN.POP.DNST",
    "Proxy_EN.GHG.CO2.MT.CE.AR5_log1p": "EN.GHG.CO2.MT.CE.AR5",
}
META_COLUMNS = ["row_uid", "Country Code", "Year", "WasteFlag", "outer_fold"]
RAW_EXCLUDED_COLUMNS = {
    *META_COLUMNS,
    "Row_ID",
    "Country Name",
    "IncomeGroup",
    "Region",
    "Waste_Generation_t",
    TARGET_LOG_COL,
}
KEEP_FEATURE_CANDIDATES = [
    "Is_AW",
    "Is_CDW",
    "Is_IW",
    "Is_MSW",
    "EG.ELC.ACCS.ZS",
    "EN.ATM.PM25.MC.M3",
    "EN.GHG.CO2.MT.CE.AR5",
    "EN.POP.DNST",
    "NV.IND.TOTL.ZS",
    "NV.SRV.TOTL.ZS",
    "NY.GDP.MKTP.KD.ZG",
    "SP.DYN.LE00.IN",
    "SP.POP.GROW",
    "SP.POP.TOTL",
    "SP.URB.GROW",
    "SP.URB.TOTL.IN.ZS",
    "NY.GDP.MKTP.PP.KD",
    "NY.GDP.PCAP.PP.KD",
]
CATEGORICAL_FEATURE_CANDIDATES = ["Is_AW", "Is_CDW", "Is_IW", "Is_MSW"]


def find_release_root(start: Path) -> Path:
    sentinel_dirs = ("Step0_Data", "Step4_ModelInputs")
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        release_candidate = candidate / "GSW_Clean_Notebook_Release"
        if all((release_candidate / name).is_dir() for name in sentinel_dirs):
            return release_candidate
    raise FileNotFoundError("Could not locate standalone release root")


In [ ]:
def _sort_timeseries_rows(frame: pd.DataFrame) -> pd.DataFrame:
    sort_cols = [col for col in ["Year", "Country Code", "WasteFlag", "row_uid"] if col in frame.columns]
    if not sort_cols:
        return frame.reset_index(drop=True)
    return frame.sort_values(sort_cols).reset_index(drop=True)


def add_waste_route_columns(frame: pd.DataFrame) -> pd.DataFrame:
    routed = frame.copy()
    if "WasteFlag" in routed.columns:
        waste_flag = routed["WasteFlag"].astype(str)
    else:
        waste_flag = pd.Series("", index=routed.index, dtype="object")
    for waste_type in WASTE_TYPES:
        routed[f"WasteRoute_{waste_type}"] = (waste_flag == waste_type).astype(int)
    return routed


def build_inner_timeseries_fold_index(train_raw: pd.DataFrame, n_splits: int = INNER_FOLDS) -> pd.DataFrame:
    sorted_train = _sort_timeseries_rows(train_raw)
    n_samples = len(sorted_train)
    if n_splits < 1:
        raise ValueError("n_splits must be at least 1")
    if n_samples < 1:
        raise ValueError("train_raw must contain at least one row")
    if n_samples < 2:
        return pd.DataFrame(
            [
                {
                    "row_uid": sorted_train.iloc[0].get("row_uid"),
                    "Country Code": sorted_train.iloc[0].get("Country Code"),
                    "Year": int(sorted_train.iloc[0]["Year"]),
                    "WasteFlag": sorted_train.iloc[0].get("WasteFlag"),
                    "inner_fold": inner_fold,
                    "split": "train" if inner_fold < n_splits else "valid",
                    "row_pos": 0,
                }
                for inner_fold in range(1, n_splits + 1)
            ]
        )

    unique_years = sorted(pd.Series(pd.to_numeric(sorted_train["Year"], errors="raise")).astype(int).unique().tolist())
    if len(unique_years) < 2:
        return pd.DataFrame(
            [
                {
                    "row_uid": sorted_train.iloc[0].get("row_uid"),
                    "Country Code": sorted_train.iloc[0].get("Country Code"),
                    "Year": int(sorted_train.iloc[0]["Year"]),
                    "WasteFlag": sorted_train.iloc[0].get("WasteFlag"),
                    "inner_fold": inner_fold,
                    "split": "train" if inner_fold < n_splits else "valid",
                    "row_pos": 0,
                }
                for inner_fold in range(1, n_splits + 1)
            ]
        )

    year_boundaries = np.linspace(0, len(unique_years), n_splits + 2, dtype=int)
    records: list[dict[str, object]] = []
    for inner_fold in range(1, n_splits + 1):
        train_year_stop = max(1, int(year_boundaries[inner_fold]))
        valid_year_stop = max(train_year_stop + 1, int(year_boundaries[inner_fold + 1]))
        valid_year_stop = min(valid_year_stop, len(unique_years))
        train_years = set(unique_years[:train_year_stop])
        valid_years = set(unique_years[train_year_stop:valid_year_stop])
        if not valid_years:
            valid_years = {unique_years[-1]}
            train_years = set(unique_years[:-1]) or {unique_years[0]}

        train_slice = sorted_train.loc[sorted_train["Year"].astype(int).isin(sorted(train_years))].copy()
        valid_slice = sorted_train.loc[sorted_train["Year"].astype(int).isin(sorted(valid_years))].copy()
        for split_name, split_df in (("train", train_slice), ("valid", valid_slice)):
            for row_pos, (_, row) in enumerate(split_df.iterrows(), start=0):
                records.append(
                    {
                        "row_uid": row.get("row_uid"),
                        "Country Code": row.get("Country Code"),
                        "Year": int(row["Year"]),
                        "WasteFlag": row.get("WasteFlag"),
                        "inner_fold": inner_fold,
                        "split": split_name,
                        "row_pos": row_pos,
                    }
                )

    return pd.DataFrame(records)


def build_full_feature_candidates(train_raw: pd.DataFrame) -> list[str]:
    candidate_columns = [col for col in train_raw.columns if col not in RAW_EXCLUDED_COLUMNS]
    return [str(col) for col in candidate_columns]


def _derive_proxy_log1p_columns(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = frame.copy()
    for proxy_col, base_col in PROXY_SOURCE_MAP.items():
        if base_col not in prepared.columns:
            continue
        base_series: pd.Series = pd.Series(pd.to_numeric(prepared[base_col], errors="coerce"), index=prepared.index)
        prepared[base_col] = base_series
        base_array = np.asarray(base_series, dtype=float)
        prepared[proxy_col] = np.log1p(np.clip(base_array, a_min=0.0, a_max=None))
    return prepared


def prepare_ten_feature_frame(frame: pd.DataFrame) -> pd.DataFrame:
    prepared = add_waste_route_columns(frame)
    prepared = _derive_proxy_log1p_columns(prepared)
    missing_formal = [col for col in FORMAL_TEN_FEATURE_CONTRACT if col not in prepared.columns]
    if missing_formal:
        raise KeyError(f"ten-feature frame is missing required formal columns: {missing_formal}")
    return prepared


def prepare_full_feature_frame(frame: pd.DataFrame, input_columns: list[str]) -> pd.DataFrame:
    prepared = frame.copy()
    if any(col in ROUTE_COLUMNS for col in input_columns):
        prepared = add_waste_route_columns(prepared)
    return prepared


def _uses_ten_feature_contract(input_columns: list[str]) -> bool:
    return list(input_columns) == get_ten_feature_input_columns()


def _prepare_model_frame(frame: pd.DataFrame, input_columns: list[str]) -> pd.DataFrame:
    if _uses_ten_feature_contract(input_columns):
        return prepare_ten_feature_frame(frame)
    return prepare_full_feature_frame(frame, input_columns)


def ensure_pycaret_sklearn_compat() -> None:
    import scipy
    import sklearn.utils
    from contextlib import contextmanager

    if hasattr(sklearn.utils, "_print_elapsed_time"):
        pass
    else:

        @contextmanager
        def _print_elapsed_time(*args, **kwargs):
            del args, kwargs
            yield

        setattr(sklearn.utils, "_print_elapsed_time", _print_elapsed_time)

    if not hasattr(scipy, "interp"):
        setattr(scipy, "interp", np.interp)


def _fit_pycaret_pipeline(
    train_frame: pd.DataFrame,
    input_columns: list[str],
    *,
    session_id: int,
    feature_selection: bool,
    keep_features: list[str] | None = None,
    categorical_features: list[str] | None = None,
):
    ensure_pycaret_sklearn_compat()
    from pycaret.regression import get_config, setup

    setup_frame = pd.DataFrame(train_frame.loc[:, input_columns + [TARGET_LOG_COL]].copy())
    setup_kwargs: dict[str, object] = {
        "data": setup_frame,
        "target": TARGET_LOG_COL,
        "imputation_type": "simple",
        "numeric_imputation": "median",
        "normalize": True,
        "remove_multicollinearity": False,
        "feature_selection": feature_selection,
        "feature_selection_method": "classic",
        "feature_selection_estimator": "lightgbm",
        "fold_strategy": "timeseries",
        "fold": INNER_FOLDS,
        "data_split_shuffle": False,
        "fold_shuffle": False,
        "train_size": 0.999999,
        "memory": False,
        "verbose": False,
        "html": False,
        "session_id": session_id,
    }
    if keep_features is not None:
        setup_kwargs["keep_features"] = list(keep_features)
    if categorical_features is not None:
        setup_kwargs["categorical_features"] = list(categorical_features)
    setup(**setup_kwargs)
    return get_config("pipeline"), get_config("data")


def _transform_with_pipeline(
    pipeline,
    frame: pd.DataFrame,
    input_columns: list[str],
    expected_columns: list[str] | None = None,
) -> pd.DataFrame:
    transformed = pipeline.transform(frame[input_columns].copy())
    if not isinstance(transformed, pd.DataFrame):
        if expected_columns is not None and len(expected_columns) == transformed.shape[1]:
            columns = expected_columns
        elif len(input_columns) == transformed.shape[1]:
            columns = input_columns
        else:
            columns = [f"feature_{idx}" for idx in range(transformed.shape[1])]
        transformed = pd.DataFrame(transformed, index=frame.index, columns=columns)
    transformed = transformed.reset_index(drop=True)
    if expected_columns is not None:
        missing_columns = [col for col in expected_columns if col not in transformed.columns]
        if missing_columns:
            raise KeyError(f"transformed frame is missing expected columns: {missing_columns}")
        transformed = transformed.loc[:, expected_columns].copy()
    if int(transformed.isna().sum().sum()) != 0:
        raise ValueError("processed feature matrix still contains NaN values")
    return transformed


def select_fold_features(train_raw: pd.DataFrame, session_id: int) -> list[str]:
    sorted_train = _sort_timeseries_rows(train_raw)
    candidates = build_full_feature_candidates(sorted_train)
    keep_features = [col for col in KEEP_FEATURE_CANDIDATES if col in candidates]
    categorical_features = [col for col in CATEGORICAL_FEATURE_CANDIDATES if col in candidates]
    pipeline, fit_data = _fit_pycaret_pipeline(
        sorted_train,
        candidates,
        session_id=session_id,
        feature_selection=True,
        keep_features=keep_features,
        categorical_features=categorical_features,
    )
    transformed = _transform_with_pipeline(pipeline, fit_data, candidates)
    return [str(col) for col in transformed.columns.tolist()]


def get_ten_feature_input_columns() -> list[str]:
    return FORMAL_TEN_FEATURE_CONTRACT + ROUTE_COLUMNS


In [ ]:
def fit_processed_view(
    train_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
    input_columns: list[str],
    expected_columns: list[str] | None = None,
    session_id: int = 0,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train_frame = _prepare_model_frame(_sort_timeseries_rows(train_raw), input_columns)
    test_frame = _prepare_model_frame(_sort_timeseries_rows(test_raw), input_columns)
    pipeline, _ = _fit_pycaret_pipeline(
        train_frame,
        input_columns,
        session_id=session_id,
        feature_selection=False,
        keep_features=input_columns,
    )
    train_processed = _transform_with_pipeline(pipeline, train_frame, input_columns, expected_columns=expected_columns)
    actual_columns = expected_columns or [str(col) for col in train_processed.columns.tolist()]
    test_processed = _transform_with_pipeline(pipeline, test_frame, input_columns, expected_columns=actual_columns)
    return train_processed, test_processed, list(actual_columns)


def write_processed_view(
    view_root: Path,
    raw_train: pd.DataFrame,
    raw_test: pd.DataFrame,
    train_processed: pd.DataFrame,
    test_processed: pd.DataFrame,
    inner_index: pd.DataFrame,
    feature_columns: list[str],
) -> None:
    processed_dir = view_root / "processed"
    processed_dir.mkdir(parents=True, exist_ok=True)
    train_out = train_processed.copy()
    test_out = test_processed.copy()
    if not set(META_COLUMNS + ["Waste_Generation_t", TARGET_LOG_COL]).issubset(train_out.columns):
        train_prefix = [col for col in [*META_COLUMNS, "Waste_Generation_t", TARGET_LOG_COL] if col in raw_train.columns]
        train_out = pd.concat([raw_train[train_prefix].reset_index(drop=True), train_out.reset_index(drop=True)], axis=1)
    if not set(META_COLUMNS + ["Waste_Generation_t", TARGET_LOG_COL]).issubset(test_out.columns):
        test_prefix = [col for col in [*META_COLUMNS, "Waste_Generation_t", TARGET_LOG_COL] if col in raw_test.columns]
        test_out = pd.concat([raw_test[test_prefix].reset_index(drop=True), test_out.reset_index(drop=True)], axis=1)
    train_out.to_csv(processed_dir / "0_train_processed.csv", index=False)
    test_out.to_csv(processed_dir / "0_test_processed.csv", index=False)
    inner_index.to_csv(processed_dir / "0_inner_timeseries_fold_index.csv", index=False)
    pd.DataFrame({"feature": feature_columns}).to_csv(processed_dir / "0_feature_columns.csv", index=False)


In [ ]:
def run_build(step4_results_root: Path) -> None:
    step4_results_root.mkdir(parents=True, exist_ok=True)

    union_features: list[str] = []
    union_seen: set[str] = set()
    fold_raw_cache: dict[int, tuple[pd.DataFrame, pd.DataFrame]] = {}

    # --- Step 1: Load raw fold assets and select full-feature candidates ---
    for outer_fold in range(1, N_OUTER_FOLDS + 1):
        raw_dir = step4_results_root / f"fold_{outer_fold}" / "raw"
        train_raw = pd.read_csv(raw_dir / "0_train.csv")
        test_raw = pd.read_csv(raw_dir / "0_test.csv")
        fold_raw_cache[outer_fold] = (train_raw, test_raw)
        selected = select_fold_features(train_raw, session_id=outer_fold)
        for column in selected:
            if column not in union_seen:
                union_seen.add(column)
                union_features.append(column)

    # --- Step 2: Write root feature manifests ---
    pd.DataFrame({"feature": union_features}).to_csv(step4_results_root / "0_union_feature_columns.csv", index=False)
    pd.DataFrame({"feature": FORMAL_TEN_FEATURE_CONTRACT}).to_csv(
        step4_results_root / "0_feature_contract_10feature.csv",
        index=False,
    )
    ten_feature_contract_reference: list[str] | None = None

    for outer_fold in range(1, N_OUTER_FOLDS + 1):
        train_raw, test_raw = fold_raw_cache[outer_fold]

        # --- Step 3: Build inner timeseries fold index ---
        inner_index = build_inner_timeseries_fold_index(train_raw, n_splits=INNER_FOLDS)

        # --- Step 4: Export full-feature unified processed views ---
        full_train, full_test, full_features = fit_processed_view(
            train_raw=train_raw,
            test_raw=test_raw,
            input_columns=build_full_feature_candidates(train_raw),
            expected_columns=union_features,
            session_id=100 + outer_fold,
        )
        write_processed_view(
            step4_results_root / f"fold_{outer_fold}" / FULL_VIEW_NAME,
            raw_train=train_raw,
            raw_test=test_raw,
            train_processed=full_train,
            test_processed=full_test,
            inner_index=inner_index,
            feature_columns=full_features,
        )

        # --- Step 5: Export ten-feature unified processed views ---
        ten_train, ten_test, ten_feature_columns = fit_processed_view(
            train_raw=train_raw,
            test_raw=test_raw,
            input_columns=get_ten_feature_input_columns(),
            expected_columns=None,
            session_id=200 + outer_fold,
        )
        if outer_fold == 1:
            ten_feature_contract_reference = ten_feature_columns
        elif ten_feature_columns != ten_feature_contract_reference:
            raise RuntimeError(f"ten_feature_unified processed contract drift on fold {outer_fold}")
        if ten_feature_contract_reference is None:
            raise RuntimeError("ten_feature_unified processed contract was not initialized")
        write_processed_view(
            step4_results_root / f"fold_{outer_fold}" / TEN_VIEW_NAME,
            raw_train=train_raw,
            raw_test=test_raw,
            train_processed=ten_train,
            test_processed=ten_test,
            inner_index=inner_index,
            feature_columns=ten_feature_contract_reference,
        )

    print(f"comparison views built under: {step4_results_root}")
    print(f"union feature count: {len(union_features)}")
    print(f"ten-feature contract size: {len(FORMAL_TEN_FEATURE_CONTRACT)}")


def main() -> None:
    release_root = find_release_root(Path.cwd().resolve())
    step4_results_root = release_root / "Step4_ModelInputs" / "2_Results"
    run_build(step4_results_root)


In [ ]:
if __name__ == "__main__":
    main()
